# Flagship Registry — v3 (English, dashboard-first, 3-level permissions)
One cell per file. Run with `streamlit run flagship_app.py`.
Default admin: `admin` / `adminflagship2026`


In [ ]:
# === flagship_app.py ===
import streamlit as st
from db import init_db, q1
from pages import (
    render_login_page,
    render_dashboard_page,
    render_list_page,
    render_flagship_detail_page,
    render_flagship_form_page,
    render_usecase_detail_page,
    render_usecase_form_page,
    render_country_form_page,
    render_users_admin_page,
    render_audit_history_page,
)

st.set_page_config(page_title="Flagship Registry", layout="wide")

init_db()

if "logged_in" not in st.session_state:
    st.session_state.logged_in = False
if "filter_countries" not in st.session_state:
    st.session_state.filter_countries = []
if "filter_divisions" not in st.session_state:
    st.session_state.filter_divisions = []
if "filter_statuses" not in st.session_state:
    st.session_state.filter_statuses = []

if not st.session_state.logged_in:
    render_login_page()
    st.stop()

is_admin = st.session_state.get("is_admin", False)

# ── Sidebar ───────────────────────────────────────────────────────────────────
with st.sidebar:
    role = "👑 Admin" if is_admin else "👤 User"
    st.write(f"{role} — **{st.session_state.username}**")

    if is_admin:
        if st.button("📈 Dashboard", use_container_width=True):
            st.session_state.page = "dashboard"
            st.rerun()
    if st.button("📂 Flagships", use_container_width=True):
        st.session_state.page = "list"
        st.rerun()
    if is_admin:
        if st.button("⚙️ Access Management", use_container_width=True):
            st.session_state.page = "users_admin"
            st.rerun()
        if st.button("📜 Activity Log", use_container_width=True):
            st.session_state.page = "audit_history"
            st.rerun()

    # Breadcrumb back-buttons
    if st.session_state.get("page") in ("flagship_detail", "flagship_form", "usecase_detail", "usecase_form", "country_form"):
        fid = st.session_state.get("current_flagship_id")
        f = q1("SELECT name FROM flagships WHERE id=?", fid) if fid else None
        if f and st.button(f"↩ {f['name']}", use_container_width=True):
            st.session_state.page = "flagship_detail"
            st.rerun()
    if st.session_state.get("page") in ("usecase_detail", "usecase_form", "country_form"):
        ucid = st.session_state.get("current_usecase_id")
        uc = q1("SELECT name FROM use_cases WHERE id=?", ucid) if ucid else None
        if uc and st.button(f"↩ {uc['name']}", use_container_width=True):
            st.session_state.page = "usecase_detail"
            st.rerun()

    st.divider()
    if st.button("🚪 Sign out", use_container_width=True):
        st.session_state.clear()
        st.rerun()

# ── Routing ───────────────────────────────────────────────────────────────────
page = st.session_state.get("page", "dashboard" if is_admin else "list")

# Guard: non-admins cannot reach admin pages
if not is_admin and page in ("dashboard", "users_admin", "audit_history"):
    page = "list"

if page == "dashboard":
    render_dashboard_page()
elif page == "list":
    render_list_page()
elif page == "flagship_detail":
    render_flagship_detail_page()
elif page == "flagship_form":
    render_flagship_form_page()
elif page == "usecase_detail":
    render_usecase_detail_page()
elif page == "usecase_form":
    render_usecase_form_page()
elif page == "country_form":
    render_country_form_page()
elif page == "users_admin":
    render_users_admin_page()
elif page == "audit_history":
    render_audit_history_page()


In [ ]:
# === db.py ===
import sqlite3
from pathlib import Path

DB_PATH = Path(__file__).parent / "flagship.db"

# IA value natures (Net Banking Impact, Gross Cost Savings, Cost Avoidance,
# Cost of Risk, RWA Savings)
VALUE_NATURES = [
    "Net Banking Impact",
    "Gross Cost Savings",
    "Cost Avoidance",
    "Cost of Risk",
    "RWA Savings",
]


def get_db():
    conn = sqlite3.connect(str(DB_PATH), check_same_thread=False)
    conn.row_factory = sqlite3.Row
    conn.execute("PRAGMA foreign_keys = ON")
    return conn


def init_db():
    from auth import hash_pw  # lazy import to avoid circular dependency
    conn = get_db()
    conn.executescript("""
    CREATE TABLE IF NOT EXISTS users (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        username TEXT UNIQUE NOT NULL,
        password_hash TEXT NOT NULL,
        is_admin INTEGER NOT NULL DEFAULT 0
    );
    CREATE TABLE IF NOT EXISTS flagships (
        id TEXT PRIMARY KEY, rank INTEGER, name TEXT NOT NULL,
        description TEXT, sponsor TEXT, owner TEXT,
        division TEXT, subdivision TEXT, poc TEXT, analyst TEXT
    );
    CREATE TABLE IF NOT EXISTS use_cases (
        id TEXT PRIMARY KEY, flagship_id TEXT NOT NULL,
        name TEXT NOT NULL, description TEXT,
        pole TEXT, entity TEXT, ai_act TEXT, technology TEXT,
        platform TEXT, brick TEXT, domain TEXT, category TEXT,
        sourcing TEXT, contributors TEXT, pilot TEXT, itsvc TEXT,
        date_itsvc TEXT, gate TEXT, date_gate TEXT
    );
    CREATE TABLE IF NOT EXISTS countries (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        use_case_id TEXT NOT NULL, country TEXT NOT NULL,
        status TEXT, prod_date TEXT,
        build REAL DEFAULT 0,
        measure TEXT, rollout TEXT, mrr_cat TEXT, mrr TEXT, owner TEXT,
        UNIQUE(use_case_id, country)
    );
    CREATE TABLE IF NOT EXISTS value_entries (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        use_case_id TEXT NOT NULL,
        country TEXT NOT NULL,
        year INTEGER NOT NULL,
        nature TEXT NOT NULL,
        amount REAL NOT NULL DEFAULT 0,
        UNIQUE(use_case_id, country, year, nature)
    );
    -- Metadata describing custom columns added by admin.
    -- field_type: 'text' | 'select' | 'date' | 'number'
    -- options: comma-separated list (for 'select')
    CREATE TABLE IF NOT EXISTS custom_fields (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        table_name TEXT NOT NULL,
        col_name TEXT NOT NULL,
        field_type TEXT NOT NULL DEFAULT 'text',
        options TEXT DEFAULT '',
        UNIQUE(table_name, col_name)
    );
    -- 3-level permissions. scope = 'flagship' | 'usecase' | 'country'
    -- ref_id holds: flagship_id, use_case_id, or country row id (as text).
    CREATE TABLE IF NOT EXISTS user_perms (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        username TEXT NOT NULL,
        scope TEXT NOT NULL,
        ref_id TEXT NOT NULL,
        UNIQUE(username, scope, ref_id)
    );
    CREATE TABLE IF NOT EXISTS audit_logs (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        timestamp DATETIME DEFAULT CURRENT_TIMESTAMP,
        username TEXT NOT NULL,
        action TEXT NOT NULL,
        table_name TEXT NOT NULL,
        record_id TEXT NOT NULL,
        details TEXT NOT NULL
    );
    """)
    conn.commit()

    if conn.execute("SELECT COUNT(*) FROM users").fetchone()[0] == 0:
        conn.execute("INSERT INTO users (username, password_hash, is_admin) VALUES (?,?,1)",
                     ("admin", hash_pw("adminflagship2026")))
        conn.commit()

    # if conn.execute("SELECT COUNT(*) FROM flagships").fetchone()[0] == 0:
    #     seed(conn)

    conn.close()


def seed(conn):
    flagships = [
        ("F01", 1, "KYC Acceleration", "Automated KYC document review and risk scoring.", "Group Compliance", "M. Lefevre", "GBIS", "Financial Crime", "A. Moreau", "C. Dubois"),
        ("F02", 2, "Credit Decision Copilot", "Generative AI copilot for credit analysts.", "GLBA", "J. Bernard", "Corporate & Investment Banking", "Credit Risk", "P. Lambert", "L. Garnier"),
        ("F03", 3, "Markets AI Trading Assistant", "Real-time AI assistant for FX and rates traders.", "Global Markets", "S. Nakamura", "Global Markets", "FX & Rates", "R. Chen", "A. Petit"),
        ("F04", 4, "Retail Customer 360", "Unified customer view and next-best-action engine.", "Retail Banking", "C. Martin", "Retail Banking", "Customer Experience", "V. Roux", "M. Faure"),
        ("F05", 5, "AML Transaction Monitoring", "Next-gen anti-money laundering alerting.", "Group Compliance", "A. Kovac", "GBIS", "AML", "N. Olsen", "T. Weiss"),
        ("F06", 6, "IT Ops AI", "AIOps platform for incident prediction and routing.", "Group IT", "D. Klein", "Group IT", "Infrastructure", "E. Tran", "O. Mendes"),
        ("F07", 7, "ESG Intelligence", "AI-driven ESG scoring and controversy detection.", "Sustainability", "I. Aliyev", "Group Strategy", "Sustainability", "B. Costa", "H. Park"),
        ("F08", 8, "Document Intelligence Hub", "Centralized document understanding service.", "Operations", "F. Boucher", "GBIS", "Operations", "K. Iyer", "D. Souza"),
        ("F09", 9, "Fraud Shield", "Real-time fraud detection across cards and transfers.", "Risk", "R. Hoffmann", "Retail Banking", "Fraud", "L. Yamada", "P. Silva"),
        ("F10", 10, "HR Talent AI", "AI-assisted CV screening and internal mobility.", "Group HR", "V. Almeida", "Group HR", "Talent", "S. Kone", "E. Ozturk"),
        ("F11", 11, "Treasury & ALM Optimizer", "AI-driven liquidity forecasting.", "Treasury", "M. Eriksson", "Finance", "ALM", "T. Andersson", "J. Lindqvist"),
        ("F12", 12, "Wealth Advisor Copilot", "Generative AI assistant for private bankers.", "Private Banking", "L. Fontaine", "Wealth Management", "Private Banking", "M. Yilmaz", "N. Rahman"),
        ("F13", 13, "Insurance Claims AI", "Automated claims triage and fraud detection.", "Insurance", "P. Novak", "Insurance", "Claims", "A. Vargas", "C. Romero"),
        ("F14", 14, "Regulatory Reporting AI", "Automated extraction of regulatory reports.", "Finance", "B. Vargas", "Finance", "Regulatory Reporting", "M. Diallo", "V. Ahmed"),
        ("F15", 15, "Cybersecurity AI", "AI-powered SOC: threat hunting, phishing detection.", "Group Security", "T. Berg", "Group IT", "Cybersecurity", "H. Mensah", "O. Karimov"),
        ("F16", 16, "Climate Stress Tests AI", "AI models for climate scenario analysis.", "Risk", "C. Iversen", "Risk", "Climate Risk", "A. Petrov", "L. Singh"),
        ("F17", 17, "Customer Service Voice AI", "Voice AI for inbound retail call centers.", "Retail Banking", "I. Renaud", "Retail Banking", "Customer Service", "D. Singh", "R. Costa"),
        ("F18", 18, "Data Governance AI", "AI for metadata enrichment and data quality.", "CDO Office", "S. Andrade", "Group Data", "Data Governance", "F. Costa", "M. Khan"),
    ]
    conn.executemany("INSERT OR IGNORE INTO flagships (id, rank, name, description, sponsor, owner, division, subdivision, poc, analyst) VALUES (?,?,?,?,?,?,?,?,?,?)", flagships)

    use_cases = [
        ("U01F01", "F01", "Document Extraction Pipeline", "Extract KYC fields from documents.", "Operations", "SG Group", "High Risk", "LLM + OCR", "Azure ML", "Document AI", "NLP", "Information Extraction", "Hybrid", "AI4Compliance", "France", "ITSVC-1042", "2024-03-15", "Gate 4", "2024-09-20"),
        ("U02F01", "F01", "Risk Scoring Engine", "PEP/sanction screening and risk scoring.", "Compliance", "SG Group", "High Risk", "Gradient Boosting", "Databricks", "Predictive AI", "Tabular ML", "Classification", "In-house", "Risk Analytics", "France", "ITSVC-1078", "2024-06-10", "Gate 3", "2025-01-12"),
        ("U01F02", "F02", "Credit Memo Summarization", "Summarize credit memos into key risk factors.", "Front Office", "GLBA", "Limited Risk", "LLM (GPT-4)", "Internal LLM Gateway", "Generative AI", "NLP", "Summarization", "Vendor", "GenAI Lab", "France", "ITSVC-1130", "2024-09-01", "Gate 4", "2025-02-28"),
        ("U01F03", "F03", "News Triage Engine", "Cluster and score market-moving news.", "Trading Floor", "GLBA Markets", "Limited Risk", "Transformers", "AWS SageMaker", "NLP Pipeline", "NLP", "Information Retrieval", "In-house", "Markets AI Squad", "France", "ITSVC-1201", "2024-11-12", "Gate 4", "2025-04-08"),
        ("U01F04", "F04", "Next-Best-Action Recommender", "Personalized product recommendations.", "Retail", "SG Retail FR", "Limited Risk", "Two-Tower DL", "GCP Vertex AI", "Recommendation", "Recsys", "Personalization", "In-house", "Personalization Lab", "France", "ITSVC-1156", "2024-08-20", "Gate 4", "2025-01-30"),
        ("U01F05", "F05", "Graph-Based Alert Triage", "GNN-based suspicious activity scoring.", "Compliance", "SG Group", "High Risk", "Graph Neural Networks", "Databricks", "Predictive AI", "Graph ML", "Anomaly Detection", "In-house", "Graph AI Team", "France", "ITSVC-1210", "2024-12-01", "Gate 3", "2025-03-22"),
        ("U01F06", "F06", "Incident Auto-Routing", "Classify and route IT tickets.", "IT Ops", "SG Group", "Minimal Risk", "BERT Classifier", "Azure ML", "NLP Classifier", "NLP", "Classification", "In-house", "AIOps Squad", "France", "ITSVC-1067", "2024-05-10", "Gate 4", "2024-10-15"),
    ]
    conn.executemany("INSERT OR IGNORE INTO use_cases (id, flagship_id, name, description, pole, entity, ai_act, technology, platform, brick, domain, category, sourcing, contributors, pilot, itsvc, date_itsvc, gate, date_gate) VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)", use_cases)

    countries_data = [
        ("U01F01", "France", "Industrialized", "2024-10-01", 1200, "Measured", "Local", "Operational Efficiency", "MRR-FR-2024-018", "M. Lefevre"),
        ("U01F01", "Italy", "Pilot", "2026-08-15", 400, "Estimated", "Local", "Operational Efficiency", "MRR-IT-2025-003", "G. Rossi"),
        ("U02F01", "France", "Pilot", "2025-02-01", 600, "Estimated", "Local", "Risk Management", "MRR-FR-2025-006", "M. Lefevre"),
        ("U01F02", "France", "Industrialized", "2025-03-01", 900, "Measured", "Global", "Productivity", "MRR-FR-2025-011", "J. Bernard"),
        ("U01F02", "Luxembourg", "Pilot", "2026-09-15", 300, "Estimated", "Local", "Productivity", "MRR-LU-2025-002", "F. Schmit"),
        ("U01F03", "France", "Industrialized", "2025-04-10", 1500, "Measured", "Global", "Revenue", "MRR-FR-2025-019", "S. Nakamura"),
        ("U01F04", "France", "Industrialized", "2025-02-15", 800, "Measured", "Local", "Revenue", "MRR-FR-2025-009", "C. Martin"),
        ("U01F05", "France", "Pilot", "2026-06-01", 700, "Estimated", "Local", "Risk Management", "MRR-FR-2025-014", "A. Kovac"),
        ("U01F06", "France", "Industrialized", "2024-11-01", 500, "Measured", "Global", "Operational Efficiency", "MRR-FR-2024-029", "D. Klein"),
    ]
    conn.executemany("INSERT OR IGNORE INTO countries (use_case_id,country,status,prod_date,build,measure,rollout,mrr_cat,mrr,owner) VALUES (?,?,?,?,?,?,?,?,?,?)", countries_data)

    value_entries = [
        ("U01F01", "France", 2025, "Gross Cost Savings", 2200),
        ("U01F01", "France", 2025, "Cost Avoidance", 600),
        ("U01F01", "France", 2026, "Gross Cost Savings", 2500),
        ("U01F01", "Italy", 2026, "Gross Cost Savings", 700),
        ("U01F01", "Italy", 2026, "Cost Avoidance", 200),
        ("U02F01", "France", 2025, "Cost of Risk", 1500),
        ("U01F02", "France", 2025, "Net Banking Impact", 1800),
        ("U01F02", "France", 2025, "Gross Cost Savings", 1400),
        ("U01F02", "Luxembourg", 2026, "Net Banking Impact", 400),
        ("U01F02", "Luxembourg", 2026, "Gross Cost Savings", 200),
        ("U01F03", "France", 2025, "Net Banking Impact", 4500),
        ("U01F04", "France", 2025, "Net Banking Impact", 3400),
        ("U01F05", "France", 2026, "Cost of Risk", 2100),
        ("U01F06", "France", 2025, "Gross Cost Savings", 1400),
    ]
    conn.executemany("INSERT OR IGNORE INTO value_entries (use_case_id,country,year,nature,amount) VALUES (?,?,?,?,?)", value_entries)
    conn.commit()


def q(sql, *args):
    conn = get_db()
    rows = conn.execute(sql, args).fetchall()
    conn.close()
    return [dict(r) for r in rows]


def q1(sql, *args):
    conn = get_db()
    r = conn.execute(sql, args).fetchone()
    conn.close()
    return dict(r) if r else None


def run(sql, *args):
    conn = get_db()
    conn.execute(sql, args)
    conn.commit()
    conn.close()


def next_f_id():
    r = q1("SELECT MAX(CAST(SUBSTR(id,2) AS INT)) n FROM flagships WHERE id LIKE 'F%'")
    return f"F{(r['n'] or 0)+1:02d}"


def next_uc_id(fid):
    n = q1("SELECT COUNT(*) n FROM use_cases WHERE flagship_id=?", fid)["n"]
    return f"U{n+1:02d}{fid}"


def get_table_columns(table_name):
    conn = get_db()
    cursor = conn.execute(f"PRAGMA table_info({table_name})")
    cols = [r["name"] for r in cursor.fetchall()]
    conn.close()
    return cols


def sanitize_col_name(name):
    import re
    s = name.strip().lower()
    s = re.sub(r'\s+', '_', s)
    s = re.sub(r'[^a-z0-9_]', '', s)
    return s


# ── Custom field metadata ─────────────────────────────────────────────────────
def add_custom_field(table_name, col_name, field_type, options=""):
    run("INSERT OR IGNORE INTO custom_fields (table_name, col_name, field_type, options) VALUES (?,?,?,?)",
        table_name, col_name, field_type, options)


def remove_custom_field(table_name, col_name):
    run("DELETE FROM custom_fields WHERE table_name=? AND col_name=?", table_name, col_name)


def get_custom_field(table_name, col_name):
    return q1("SELECT * FROM custom_fields WHERE table_name=? AND col_name=?", table_name, col_name)


def get_custom_fields(table_name):
    return q("SELECT * FROM custom_fields WHERE table_name=? ORDER BY id", table_name)


# ── IA value helpers ──────────────────────────────────────────────────────────
def get_value_entries(use_case_id=None, country=None, year=None):
    sql = "SELECT * FROM value_entries WHERE 1=1"
    params = []
    if use_case_id is not None:
        sql += " AND use_case_id=?"
        params.append(use_case_id)
    if country is not None:
        sql += " AND country=?"
        params.append(country)
    if year is not None:
        sql += " AND year=?"
        params.append(year)
    sql += " ORDER BY year, nature"
    return q(sql, *params)


def usecase_total_value(use_case_id):
    r = q1("SELECT COALESCE(SUM(amount),0) n FROM value_entries WHERE use_case_id=?", use_case_id)
    return r["n"] if r else 0


def usecase_value_by_nature(use_case_id):
    rows = q("""
        SELECT nature, COALESCE(SUM(amount),0) total
        FROM value_entries WHERE use_case_id=?
        GROUP BY nature ORDER BY total DESC
    """, use_case_id)
    return {r["nature"]: r["total"] for r in rows}


def country_total_value(use_case_id, country):
    r = q1("SELECT COALESCE(SUM(amount),0) n FROM value_entries WHERE use_case_id=? AND country=?", use_case_id, country)
    return r["n"] if r else 0


In [ ]:
# === auth.py ===
import hashlib
from db import q, q1, run


def hash_pw(pw):
    return hashlib.sha256(pw.encode()).hexdigest()


def check_login(u, p):
    return bool(q1("SELECT 1 FROM users WHERE username=? AND password_hash=?", u, hash_pw(p)))


def is_user_admin(username):
    r = q1("SELECT is_admin FROM users WHERE username=?", username)
    return bool(r and r["is_admin"])


def create_user(u, p, is_admin=False):
    try:
        run("INSERT INTO users (username, password_hash, is_admin) VALUES (?,?,?)",
            u, hash_pw(p), 1 if is_admin else 0)
        return True
    except Exception:
        return False


# ── 3-level permission model ──────────────────────────────────────────────────
# A user can be granted access at three scopes:
#   - 'flagship' (ref_id = flagship_id)  -> covers the flagship AND all its
#       use cases and all their country deployments.
#   - 'usecase'  (ref_id = use_case_id)  -> covers that use case AND all its
#       country deployments.
#   - 'country'  (ref_id = country row id) -> that single deployment only.
# Granting a flagship therefore implicitly grants everything below it
# ("give access to a flagship, leave use case / country empty => access to the
#  flagship and all its use cases").

def get_user_perms(username):
    return q("SELECT scope, ref_id FROM user_perms WHERE username=?", username)


def set_user_perms(username, perms):
    """perms = list of (scope, ref_id). Replaces all existing perms."""
    run("DELETE FROM user_perms WHERE username=?", username)
    for scope, ref_id in perms:
        run("INSERT OR IGNORE INTO user_perms (username, scope, ref_id) VALUES (?,?,?)",
            username, scope, str(ref_id))


def _flagship_perms(username):
    return {p["ref_id"] for p in q("SELECT ref_id FROM user_perms WHERE username=? AND scope='flagship'", username)}


def _usecase_perms(username):
    return {p["ref_id"] for p in q("SELECT ref_id FROM user_perms WHERE username=? AND scope='usecase'", username)}


def _country_perms(username):
    return {p["ref_id"] for p in q("SELECT ref_id FROM user_perms WHERE username=? AND scope='country'", username)}


def is_flagship_authorized(username, flagship_id):
    if is_user_admin(username):
        return True
    # direct flagship grant
    if flagship_id in _flagship_perms(username):
        return True
    # any usecase grant under this flagship
    if q1("""SELECT 1 FROM user_perms p JOIN use_cases uc ON p.ref_id = uc.id
             WHERE p.username=? AND p.scope='usecase' AND uc.flagship_id=?""", username, flagship_id):
        return True
    # any country grant under this flagship
    if q1("""SELECT 1 FROM user_perms p
             JOIN countries c ON CAST(p.ref_id AS INTEGER) = c.id
             JOIN use_cases uc ON c.use_case_id = uc.id
             WHERE p.username=? AND p.scope='country' AND uc.flagship_id=?""", username, flagship_id):
        return True
    return False


def is_usecase_authorized(username, usecase_id):
    if is_user_admin(username):
        return True
    uc = q1("SELECT flagship_id FROM use_cases WHERE id=?", usecase_id)
    if uc and uc["flagship_id"] in _flagship_perms(username):
        return True
    if usecase_id in _usecase_perms(username):
        return True
    if q1("""SELECT 1 FROM user_perms p
             JOIN countries c ON CAST(p.ref_id AS INTEGER) = c.id
             WHERE p.username=? AND p.scope='country' AND c.use_case_id=?""", username, usecase_id):
        return True
    return False


def is_country_authorized(username, country_id):
    if is_user_admin(username):
        return True
    c = q1("SELECT id, use_case_id FROM countries WHERE id=?", country_id)
    if not c:
        return False
    uc = q1("SELECT flagship_id FROM use_cases WHERE id=?", c["use_case_id"])
    if uc and uc["flagship_id"] in _flagship_perms(username):
        return True
    if c["use_case_id"] in _usecase_perms(username):
        return True
    if str(country_id) in _country_perms(username):
        return True
    return False


def authorized_flagship_ids(username):
    """Return set of flagship ids the user can see (any scope)."""
    if is_user_admin(username):
        return {r["id"] for r in q("SELECT id FROM flagships")}
    ids = set(_flagship_perms(username))
    for r in q("""SELECT DISTINCT uc.flagship_id fid FROM user_perms p
                  JOIN use_cases uc ON p.ref_id = uc.id
                  WHERE p.username=? AND p.scope='usecase'""", username):
        ids.add(r["fid"])
    for r in q("""SELECT DISTINCT uc.flagship_id fid FROM user_perms p
                  JOIN countries c ON CAST(p.ref_id AS INTEGER) = c.id
                  JOIN use_cases uc ON c.use_case_id = uc.id
                  WHERE p.username=? AND p.scope='country'""", username):
        ids.add(r["fid"])
    return ids


def log_change(username, action, table_name, record_id, details):
    run("""
        INSERT INTO audit_logs (username, action, table_name, record_id, details)
        VALUES (?,?,?,?,?)
    """, username, action, table_name, str(record_id), details)


In [ ]:
# === ui_components.py ===
import streamlit as st
from db import VALUE_NATURES, get_custom_field

COUNTRIES = ['France', 'Italy', 'Belgium', 'Luxembourg', 'Poland', 'Czech Republic', 'Romania',
             'Morocco', 'Algeria', 'Tunisia', 'Germany', 'Spain', 'UK', 'Switzerland', 'Netherlands', 'USA', 'Singapore', 'Japan']
AI_ACT    = ["Minimal Risk", "Limited Risk", "High Risk", "Unacceptable Risk"]
STATUS    = ["Draft", "Pilot", "Industrialized"]
MEASURE   = ["Estimated", "Measured"]
ROLLOUT   = ["Local", "Regional", "Global"]
GATES     = ["Gate 1", "Gate 2", "Gate 3", "Gate 4", "Gate 5"]
SOURCING  = ["In-house", "Vendor", "Hybrid"]
PLATFORMS = ["Azure ML", "GCP Vertex AI", "AWS SageMaker", "Databricks", "Internal LLM Gateway", "On-prem", "Azure", "AWS", "Other"]
MRR_CATS  = ["", "Operational Efficiency", "Risk Management", "Revenue", "Productivity", "Cost Reduction"]

PROTECTED_COLS = {
    "flagships": [
        "id", "rank", "name", "description", "sponsor", "owner", 
        "division", "subdivision", "poc", "analyst"
    ],
    "use_cases": [
        "id", "flagship_id", "name", "description", "pole", "entity", 
        "ai_act", "technology", "platform", "brick", "domain", 
        "category", "sourcing", "contributors", "pilot", "itsvc", 
        "date_itsvc", "gate", "date_gate"
    ],
    "countries": [
        "id", "use_case_id", "country", "status", "prod_date", 
        "build", "measure", "rollout", "mrr_cat", "mrr", "owner"
    ]
}

# Built-in select fields (native columns with fixed option lists)
SELECT_BOXES = {
    "ai_act": AI_ACT,
    "sourcing": SOURCING,
    "gate": GATES,
    "status": STATUS,
    "measure": MEASURE,
    "rollout": ROLLOUT,
    "mrr_cat": MRR_CATS
}

MONEY_COLS = ("build",)

FIELD_TYPES = ["text", "select", "date", "number"]

# Pretty labels for known columns (English)
LABELS = {
    "ai_act": "AI Act", "poc": "POC", "itsvc": "ITSVC", "date_itsvc": "ITSVC Date",
    "date_gate": "Gate Date", "mrr_cat": "MRR Category", "mrr": "MRR",
    "prod_date": "Production Date", "val_capped": "Capped Value",
    "ia_value": "IA Value", "build": "Build",
}


def nice_label(col_name):
    return LABELS.get(col_name, col_name.replace("_", " ").title())


def fmt_money(v):
    try:
        return f"{int(round(float(v))):,} k€".replace(",", " ")
    except (TypeError, ValueError):
        return "—"


def render_field_input(table_name, col_name, value, key_prefix):
    """Render the right widget for a column, honoring custom field types."""
    # Custom field defined by admin? Use its declared type.
    cf = get_custom_field(table_name, col_name)
    if cf:
        ft = cf["field_type"]
        if ft == "select":
            opts = [o.strip() for o in (cf["options"] or "").split(",") if o.strip()]
            opts = [""] + opts if "" not in opts else opts
            try:
                idx = opts.index(value) if value in opts else 0
            except ValueError:
                idx = 0
            return st.selectbox(nice_label(col_name), opts, index=idx, key=f"{key_prefix}_{col_name}")
        elif ft == "date":
            return st.text_input(nice_label(col_name) + " (YYYY-MM-DD)", value=str(value) if value else "", key=f"{key_prefix}_{col_name}")
        elif ft == "number":
            try:
                v = float(value) if value not in (None, "") else 0.0
            except (TypeError, ValueError):
                v = 0.0
            return st.number_input(nice_label(col_name), value=v, key=f"{key_prefix}_{col_name}")
        else:
            return st.text_input(nice_label(col_name), value=str(value) if value is not None else "", key=f"{key_prefix}_{col_name}")

    # Native fields
    if col_name in SELECT_BOXES:
        options = SELECT_BOXES[col_name]
        try:
            idx = options.index(value)
        except ValueError:
            idx = 0
        return st.selectbox(nice_label(col_name), options, index=idx, key=f"{key_prefix}_{col_name}")
    elif col_name == "description":
        return st.text_area("Description", value=value or "", key=f"{key_prefix}_{col_name}")
    elif col_name in MONEY_COLS:
        return st.number_input(nice_label(col_name) + " (k€)", min_value=0, value=int(value or 0), key=f"{key_prefix}_{col_name}")
    elif col_name in ("prod_date", "date_itsvc", "date_gate"):
        return st.text_input(nice_label(col_name) + " (YYYY-MM-DD)", value=str(value) if value else "", key=f"{key_prefix}_{col_name}")
    elif col_name == "rank":
        return st.number_input("Rank", min_value=1, max_value=99, value=int(value or 1), key=f"{key_prefix}_{col_name}")
    else:
        return st.text_input(nice_label(col_name), value=str(value) if value is not None else "", key=f"{key_prefix}_{col_name}")


def render_detail_grid(row, exclude_cols=None, n_cols=2):
    """Clean read-only grid of field/value pairs."""
    if exclude_cols is None:
        exclude_cols = []
    cols = [k for k in row.keys() if k not in exclude_cols]
    if not cols:
        return
    columns = st.columns(n_cols)
    for i, col_name in enumerate(cols):
        val = row[col_name]
        if val is None or val == "":
            display_val = "—"
        elif isinstance(val, (int, float)) and col_name in MONEY_COLS:
            display_val = fmt_money(val)
        else:
            display_val = str(val)
        with columns[i % n_cols]:
            st.markdown(f"<div style='font-size:0.78rem;color:#888;text-transform:uppercase;letter-spacing:0.04em'>{nice_label(col_name)}</div>"
                        f"<div style='font-size:0.95rem;margin-bottom:0.6rem'>{display_val}</div>",
                        unsafe_allow_html=True)


In [ ]:
# === quarters.py ===
"""
Quarter allocation logic for IA value.

Rule (per latest spec):
  - The annual value is ALWAYS divided by 4 (one quarter share = annual / 4).
  - For the PRODUCTION YEAR: only quarters >= the production quarter are filled.
    Quarters before go-live stay at 0 and that share is simply lost.
  - For years AFTER the production year: all 4 quarters get annual / 4.
  - For years BEFORE the production year: everything is 0.

Example: annual = 100, prod in 2026-Q3
         => 2026: Q1=0, Q2=0, Q3=25, Q4=25   (effective total 50)
         => 2027: Q1=25, Q2=25, Q3=25, Q4=25 (effective total 100)
"""
from datetime import datetime


def quarter_from_date(date_str):
    """Return quarter number (1-4) from a 'YYYY-MM-DD' date, or None."""
    if not date_str:
        return None
    for fmt in ("%Y-%m-%d", "%Y/%m/%d", "%d/%m/%Y"):
        try:
            d = datetime.strptime(str(date_str).strip(), fmt)
            return (d.month - 1) // 3 + 1
        except (ValueError, TypeError):
            continue
    return None


def year_from_date(date_str):
    """Return the year (int) from a date, or None."""
    if not date_str:
        return None
    for fmt in ("%Y-%m-%d", "%Y/%m/%d", "%d/%m/%Y"):
        try:
            return datetime.strptime(str(date_str).strip(), fmt).year
        except (ValueError, TypeError):
            continue
    return None


def split_annual_to_quarters(annual_value, prod_date, target_year):
    """
    Spread `annual_value` across the 4 quarters of `target_year`.

    Always uses per_quarter = annual_value / 4.
    - prod year  : fill only quarters >= prod quarter (rest lost).
    - after prod : fill all 4 quarters.
    - before prod: all zeros.
    - no prod date: fill all 4 quarters (assume full-year).

    Returns [q1, q2, q3, q4].
    """
    annual_value = float(annual_value or 0)
    per_q = annual_value / 4
    prod_year = year_from_date(prod_date)
    prod_q = quarter_from_date(prod_date)

    if prod_year is None or prod_q is None:
        return [per_q, per_q, per_q, per_q]

    if target_year < prod_year:
        return [0.0, 0.0, 0.0, 0.0]

    if target_year > prod_year:
        return [per_q, per_q, per_q, per_q]

    quarters = [0.0, 0.0, 0.0, 0.0]
    for qi in range(prod_q, 5):
        quarters[qi - 1] = per_q
    return quarters


def quarters_for_country_year(value_entries, prod_date, target_year):
    """
    Aggregate value entries (each {amount, nature, year}) for a target year.
    Returns dict with total_annual, quarters, by_nature, by_nature_quarters.
    """
    by_nature = {}
    for e in value_entries:
        if int(e.get("year") or 0) != int(target_year):
            continue
        nat = e.get("nature") or "Other"
        by_nature[nat] = by_nature.get(nat, 0.0) + float(e.get("amount") or 0)

    by_nature_quarters = {}
    total_q = [0.0, 0.0, 0.0, 0.0]
    for nat, amt in by_nature.items():
        qs = split_annual_to_quarters(amt, prod_date, target_year)
        by_nature_quarters[nat] = qs
        for i in range(4):
            total_q[i] += qs[i]

    return {
        "total_annual": sum(by_nature.values()),
        "quarters": total_q,
        "by_nature": by_nature,
        "by_nature_quarters": by_nature_quarters,
    }


In [ ]:
# === pages.py ===
import streamlit as st
from db import (q, q1, run, next_f_id, next_uc_id, get_table_columns, sanitize_col_name,
                VALUE_NATURES, get_value_entries, usecase_total_value, usecase_value_by_nature,
                country_total_value, add_custom_field, remove_custom_field, get_custom_fields)
from auth import (is_flagship_authorized, is_usecase_authorized, is_country_authorized,
                  log_change, check_login, create_user, is_user_admin,
                  get_user_perms, set_user_perms, authorized_flagship_ids)
from ui_components import (PROTECTED_COLS, COUNTRIES, FIELD_TYPES, render_field_input,
                           render_detail_grid, fmt_money, nice_label)
from quarters import split_annual_to_quarters


def _is_admin():
    return st.session_state.get("is_admin", False)


def _uname():
    return st.session_state.username


# ── Popups / Dialogs for deletion confirmations ────────────────────────────────
@st.dialog("Delete Flagship")
def delete_flagship_dialog(fid, f_name):
    n_uc = q1("SELECT COUNT(*) n FROM use_cases WHERE flagship_id=?", fid)["n"]
    n_c = q1("SELECT COUNT(*) n FROM countries c JOIN use_cases uc ON c.use_case_id=uc.id WHERE uc.flagship_id=?", fid)["n"]
    n_v = q1("SELECT COUNT(*) n FROM value_entries ve JOIN use_cases uc ON ve.use_case_id=uc.id WHERE uc.flagship_id=?", fid)["n"]
    st.error(f"⚠️ **PERMANENT DELETE** ⚠️\n\nDeleting **{f_name}** will permanently erase from the database:\n"
             f"- **{n_uc}** use case(s)\n- **{n_c}** country deployment(s)\n- **{n_v}** IA value entrie(s)\n\n**This cannot be undone.**")
    confirm = st.text_input("Type DELETE to confirm", key="confirm_txt_f_dlg")
    cc1, cc2 = st.columns(2)
    with cc1:
        if st.button("Yes, delete everything", type="primary", disabled=(confirm != "DELETE"), key="confirm_del_f_btn"):
            for ucid in [r["id"] for r in q("SELECT id FROM use_cases WHERE flagship_id=?", fid)]:
                run("DELETE FROM value_entries WHERE use_case_id=?", ucid)
                run("DELETE FROM countries WHERE use_case_id=?", ucid)
            run("DELETE FROM use_cases WHERE flagship_id=?", fid)
            log_change(_uname(), "DELETE", "flagships", fid, f"Deleted flagship {f_name} (+{n_uc} UC, {n_c} countries, {n_v} values)")
            run("DELETE FROM flagships WHERE id=?", fid)
            st.session_state.page = "list"
            st.rerun()
    with cc2:
        if st.button("Cancel", key="cancel_del_f_btn"):
            st.rerun()


@st.dialog("Delete Use Case")
def delete_usecase_dialog(ucid, uc_name):
    n_c = q1("SELECT COUNT(*) n FROM countries WHERE use_case_id=?", ucid)["n"]
    n_v = q1("SELECT COUNT(*) n FROM value_entries WHERE use_case_id=?", ucid)["n"]
    st.error(f"⚠️ **PERMANENT DELETE** ⚠️\n\nDeleting **{uc_name}** will erase **{n_c}** deployment(s) and **{n_v}** value entrie(s). Irreversible.")
    confirm = st.text_input("Type DELETE to confirm", key="confirm_txt_uc_dlg")
    cc1, cc2 = st.columns(2)
    with cc1:
        if st.button("Yes, delete", type="primary", disabled=(confirm != "DELETE"), key="confirm_del_uc_btn"):
            run("DELETE FROM value_entries WHERE use_case_id=?", ucid)
            run("DELETE FROM countries WHERE use_case_id=?", ucid)
            log_change(_uname(), "DELETE", "use_cases", ucid, f"Deleted use case {uc_name}")
            run("DELETE FROM use_cases WHERE id=?", ucid)
            st.session_state.page = "flagship_detail"
            st.rerun()
    with cc2:
        if st.button("Cancel", key="cancel_del_uc_btn"):
            st.rerun()


@st.dialog("Delete Country Deployment")
def delete_country_dialog(ucid, country):
    n_v = q1("SELECT COUNT(*) n FROM value_entries WHERE use_case_id=? AND country=?", ucid, country)["n"]
    st.error(f"⚠️ Deleting **{country}** also erases **{n_v}** value entrie(s). Irreversible.")
    confirm = st.text_input("Type DELETE to confirm", key="confirm_txt_c_dlg")
    cc1, cc2 = st.columns(2)
    with cc1:
        if st.button("Confirm delete", type="primary", disabled=(confirm != "DELETE"), key="confirm_del_c_btn"):
            run("DELETE FROM value_entries WHERE use_case_id=? AND country=?", ucid, country)
            log_change(_uname(), "DELETE", "countries", f"{ucid}/{country}", f"Deleted deployment {country}")
            run("DELETE FROM countries WHERE use_case_id=? AND country=?", ucid, country)
            st.rerun()
    with cc2:
        if st.button("Cancel", key="cancel_del_c_btn"):
            st.rerun()


@st.dialog("Delete Custom Field")
def delete_custom_field_dialog(table_name, col_name):
    st.warning(f"⚠️ Are you sure you want to permanently delete the custom field '{col_name}' from table '{table_name}'?")
    st.write("This will permanently delete all data stored in this column for all records. This action cannot be undone.")
    confirm = st.text_input("Type DELETE to confirm", key="confirm_txt_cf_dlg")
    cc1, cc2 = st.columns(2)
    with cc1:
        if st.button("Yes, delete permanently", type="primary", disabled=(confirm != "DELETE"), key="confirm_del_cf_btn"):
            try:
                run(f"ALTER TABLE {table_name} DROP COLUMN {col_name}")
                remove_custom_field(table_name, col_name)
                log_change(_uname(), "SCHEMA_DROP", table_name, col_name, f"Dropped column {col_name}")
                st.rerun()
            except Exception as e:
                st.error(f"Error dropping column: {e}")
    with cc2:
        if st.button("Cancel", key="cancel_del_cf_btn"):
            st.rerun()


@st.dialog("Delete Value Entry")
def delete_value_entry_dialog(ve_id, yr, nat, amt, target):
    st.warning(f"⚠️ Are you sure you want to delete the value entry for **{yr}** - **{nat}** ({fmt_money(amt)})?")
    cc1, cc2 = st.columns(2)
    with cc1:
        if st.button("Yes, delete", type="primary", key="confirm_del_ve_btn"):
            run("DELETE FROM value_entries WHERE id=?", ve_id)
            log_change(_uname(), "DELETE", "value_entries", ve_id, f"Deleted value {yr}/{nat} ({target})")
            st.rerun()
    with cc2:
        if st.button("Cancel", key="cancel_del_ve_btn"):
            st.rerun()


@st.dialog("Delete User")
def delete_user_dialog(username):
    st.warning(f"⚠️ Are you sure you want to delete user **{username}**? This will remove all their access permissions.")
    cc1, cc2 = st.columns(2)
    with cc1:
        if st.button("Yes, delete user", type="primary", key="confirm_del_usr_btn"):
            run("DELETE FROM users WHERE username=?", username)
            set_user_perms(username, [])
            log_change(_uname(), "DELETE", "users", username, f"Deleted user {username}")
            st.rerun()
    with cc2:
        if st.button("Cancel", key="cancel_del_usr_btn"):
            st.rerun()


@st.dialog("Clear Activity Log")
def clear_audit_logs_dialog():
    st.error("⚠️ **CRITICAL WARNING** ⚠️\n\nAre you sure you want to permanently clear the activity log? All audit trails will be deleted forever.")
    confirm = st.text_input("Type CLEAR to confirm", key="confirm_txt_audit_dlg")
    cc1, cc2 = st.columns(2)
    with cc1:
        if st.button("Yes, clear everything", type="primary", disabled=(confirm != "CLEAR"), key="confirm_clear_audit_btn"):
            run("DELETE FROM audit_logs")
            st.rerun()
    with cc2:
        if st.button("Cancel", key="cancel_clear_audit_btn"):
            st.rerun()


# ── small UI helpers ──────────────────────────────────────────────────────────
def card_open(title, subtitle="", accent="#2563eb"):
    st.markdown(
        f"<div style='border:1px solid #e5e7eb;border-left:4px solid {accent};"
        f"border-radius:10px;padding:14px 16px;margin-bottom:10px;background:#fafafa'>"
        f"<div style='font-weight:600;font-size:1.02rem'>{title}</div>"
        f"<div style='color:#6b7280;font-size:0.85rem'>{subtitle}</div></div>",
        unsafe_allow_html=True)


def status_badge(status):
    colors = {"Industrialized": "#16a34a", "Pilot": "#d97706", "Draft": "#6b7280"}
    c = colors.get(status, "#6b7280")
    return f"<span style='background:{c};color:white;padding:2px 8px;border-radius:10px;font-size:0.72rem'>{status or '—'}</span>"


def log_record_update(table_name, record_id, new_data, query_old_sql, *query_old_params):
    """
    Compare old record values with new_data and log a detailed UPDATE entry in audit_logs.
    """
    old_row = q1(query_old_sql, *query_old_params)
    if not old_row:
        log_change(_uname(), "UPDATE", table_name, record_id, f"Updated {table_name[:-1] if table_name.endswith('s') else table_name}")
        return

    diffs = []
    for key, new_val in new_data.items():
        if key in ("id", "flagship_id", "use_case_id", "country"):
            continue
        old_val = old_row.get(key)
        str_old = str(old_val if old_val is not None else "").strip()
        str_new = str(new_val if new_val is not None else "").strip()
        if str_old != str_new:
            lbl = nice_label(key)
            diffs.append(f"{lbl}: {str_old or '—'} → {str_new or '—'}")

    if diffs:
        details = f"Modified: " + ", ".join(diffs)
        log_change(_uname(), "UPDATE", table_name, record_id, details)


def render_local_history(table_name, record_id, child_records=None):
    """
    Render a clean list of history entries for this specific record (and optional children).
    """
    st.write("---")
    st.subheader("📜 Modification History")

    sql = "SELECT timestamp, username, action, details FROM audit_logs WHERE (table_name=? AND record_id=?)"
    params = [table_name, str(record_id)]

    if child_records:
        for c_table, c_id in child_records:
            sql += " OR (table_name=? AND record_id=?)"
            params.extend([c_table, str(c_id)])
            
    sql += " ORDER BY timestamp DESC LIMIT 20"
    logs = q(sql, *params)

    if not logs:
        st.info("No modifications recorded for this item yet.")
    else:
        emojis = {"INSERT": "➕", "UPDATE": "📝", "UPSERT": "💾", "DELETE": "🗑️", "SCHEMA_ADD": "⚙️", "SCHEMA_DROP": "⚙️"}
        for log in logs:
            e = emojis.get(log["action"], "🔔")
            t = log["timestamp"]
            details = log["details"]
            st.markdown(
                f"<div style='font-size:0.88rem; padding: 4px 0; border-bottom: 1px solid #f3f4f6;'>"
                f"<span style='color:#6b7280; font-family:monospace;'>[{t}]</span> "
                f"<b>{e} {log['action']}</b> by <i>{log['username']}</i>: {details}"
                f"</div>",
                unsafe_allow_html=True
            )


# ──────────────────────────────────────────────────────────────────────────────
def render_login_page():
    st.title("Flagship Registry")
    st.subheader("Sign in")
    u = st.text_input("Username", key="li_u")
    p = st.text_input("Password", type="password", key="li_p")
    if st.button("Sign in", type="primary"):
        if check_login(u, p):
            st.session_state.logged_in = True
            st.session_state.username = u
            st.session_state.is_admin = is_user_admin(u)
            # admins land on the dashboard, users on the list
            st.session_state.page = "dashboard" if st.session_state.is_admin else "list"
            st.rerun()
        else:
            st.error("Invalid credentials.")


# ── DASHBOARD (admin landing page) ────────────────────────────────────────────
def render_dashboard_page():
    if not _is_admin():
        st.error("Admin only.")
        st.stop()

    st.title("📈 Dashboard")
    st.caption("Annual & quarterly view of gains and costs, based on each deployment's production date.")

    all_years = [r["year"] for r in q("SELECT DISTINCT year FROM value_entries ORDER BY year")]
    if not all_years:
        st.info("No IA value recorded yet.")
        st.stop()

    # ── Filters ───────────────────────────────────────────────────────────────
    st.markdown("##### Filters")
    r1c1, r1c2, r1c3 = st.columns(3)
    with r1c1:
        sel_year = st.selectbox("Year", all_years, index=len(all_years) - 1)
    with r1c2:
        fl_opts = {r["id"]: f"#{r['rank']} {r['name']}" for r in q("SELECT id, rank, name FROM flagships ORDER BY rank")}
        f_flag = st.multiselect("Flagship", list(fl_opts.keys()), format_func=lambda x: fl_opts[x])
    with r1c3:
        # use cases, optionally constrained by selected flagships
        if f_flag:
            uc_rows = q("SELECT id, name FROM use_cases WHERE flagship_id IN ({}) ORDER BY id".format(
                ",".join(["?"] * len(f_flag))), *f_flag)
        else:
            uc_rows = q("SELECT id, name FROM use_cases ORDER BY id")
        uc_opts = {r["id"]: r["name"] for r in uc_rows}
        f_uc = st.multiselect("Use case", list(uc_opts.keys()), format_func=lambda x: uc_opts[x])

    r2c1, r2c2 = st.columns(2)
    with r2c1:
        all_ctry = [r["country"] for r in q("SELECT DISTINCT country FROM countries ORDER BY country")]
        f_ctry = st.multiselect("Country", all_ctry)
    with r2c2:
        f_nat = st.multiselect("Nature", VALUE_NATURES)

    active_filters = [x for x in [
        ("Year", str(sel_year)),
        ("Flagship", ", ".join(fl_opts[i] for i in f_flag) if f_flag else None),
        ("Use case", ", ".join(uc_opts[i] for i in f_uc) if f_uc else None),
        ("Country", ", ".join(f_ctry) if f_ctry else None),
        ("Nature", ", ".join(f_nat) if f_nat else None),
    ] if x[1]]
    if active_filters:
        chips = "  ".join(
            f"<span style='background:#2563eb;color:white;padding:3px 10px;border-radius:12px;"
            f"font-size:0.75rem;margin-right:4px'>{k}: {v}</span>" for k, v in active_filters)
        st.markdown("**Active filters:** " + chips, unsafe_allow_html=True)

    # ── Query value entries with filters ──────────────────────────────────────
    sql = """
        SELECT ve.use_case_id, ve.country, ve.year, ve.nature, ve.amount,
               c.prod_date, c.status, c.build, f.division, uc.flagship_id, uc.name uc_name, f.name fl_name
        FROM value_entries ve
        JOIN use_cases uc ON ve.use_case_id = uc.id
        JOIN flagships f ON uc.flagship_id = f.id
        JOIN countries c ON c.use_case_id = uc.id AND c.country = ve.country
        WHERE ve.year = ?
    """
    params = [sel_year]
    if f_flag:
        sql += " AND uc.flagship_id IN ({})".format(",".join(["?"] * len(f_flag)))
        params += f_flag
    if f_uc:
        sql += " AND ve.use_case_id IN ({})".format(",".join(["?"] * len(f_uc)))
        params += f_uc
    if f_ctry:
        sql += " AND ve.country IN ({})".format(",".join(["?"] * len(f_ctry)))
        params += f_ctry
    if f_nat:
        sql += " AND ve.nature IN ({})".format(",".join(["?"] * len(f_nat)))
        params += f_nat
    rows = q(sql, *params)

    total_annual = 0.0
    quarters = [0.0, 0.0, 0.0, 0.0]
    by_nature = {}
    by_nature_q = {}
    by_flag = {}
    for r in rows:
        total_annual += float(r["amount"])
        by_nature[r["nature"]] = by_nature.get(r["nature"], 0.0) + float(r["amount"])
        by_flag[r["fl_name"]] = by_flag.get(r["fl_name"], 0.0) + float(r["amount"])
        qs = split_annual_to_quarters(r["amount"], r["prod_date"], sel_year)
        for i in range(4):
            quarters[i] += qs[i]
            by_nature_q.setdefault(r["nature"], [0, 0, 0, 0])[i] += qs[i]

    # Build cost (respect flagship/usecase/country filters, not nature)
    bsql = """SELECT COALESCE(SUM(c.build),0) n FROM countries c
              JOIN use_cases uc ON c.use_case_id = uc.id WHERE 1=1"""
    bparams = []
    if f_flag:
        bsql += " AND uc.flagship_id IN ({})".format(",".join(["?"] * len(f_flag)))
        bparams += f_flag
    if f_uc:
        bsql += " AND c.use_case_id IN ({})".format(",".join(["?"] * len(f_uc)))
        bparams += f_uc
    if f_ctry:
        bsql += " AND c.country IN ({})".format(",".join(["?"] * len(f_ctry)))
        bparams += f_ctry
    total_build = q1(bsql, *bparams)["n"]

    st.divider()
    m1, m2, m3, m4 = st.columns(4)
    m1.metric(f"Gains {sel_year}", fmt_money(total_annual))
    m2.metric("Build (cost)", fmt_money(total_build))
    m3.metric("Net", fmt_money(total_annual - total_build))
    m4.metric("Deployments", len({(r['use_case_id'], r['country']) for r in rows}))

    # ── Quarterly gains ───────────────────────────────────────────────────────
    st.subheader(f"Quarterly gains — {sel_year}")
    qc = st.columns(4)
    for i, lbl in enumerate(["Q1", "Q2", "Q3", "Q4"]):
        qc[i].metric(lbl, fmt_money(quarters[i]))
    try:
        import pandas as pd
        st.bar_chart(pd.DataFrame({"Quarter": ["Q1", "Q2", "Q3", "Q4"], "Gains (k€)": quarters}).set_index("Quarter"))
    except Exception:
        pass

    # ── Breakdown by nature ───────────────────────────────────────────────────
    st.subheader("Breakdown by nature")
    if by_nature:
        try:
            import pandas as pd
            data = []
            for nat, amt in by_nature.items():
                qn = by_nature_q.get(nat, [0, 0, 0, 0])
                data.append({"Nature": nat, "Annual": amt, "Q1": qn[0], "Q2": qn[1], "Q3": qn[2], "Q4": qn[3]})
            df = pd.DataFrame(data).sort_values("Annual", ascending=False)
            st.dataframe(df.style.format({c: "{:,.0f}".format for c in ["Annual", "Q1", "Q2", "Q3", "Q4"]}),
                         use_container_width=True, hide_index=True)
        except Exception:
            for nat, amt in sorted(by_nature.items(), key=lambda x: -x[1]):
                st.write(f"**{nat}** : {fmt_money(amt)}")
    else:
        st.info("No data for these filters.")

    # ── Breakdown by flagship ─────────────────────────────────────────────────
    if by_flag:
        st.subheader("Gains by flagship")
        try:
            import pandas as pd
            df2 = pd.DataFrame([{"Flagship": k, "Gains (k€)": v} for k, v in by_flag.items()]).sort_values("Gains (k€)", ascending=False)
            st.dataframe(df2.style.format({"Gains (k€)": "{:,.0f}".format}), use_container_width=True, hide_index=True)
        except Exception:
            pass

    st.divider()
    if st.button("📂 Browse flagships"):
        st.session_state.page = "list"
        st.rerun()


# ── LIST (flagships) ──────────────────────────────────────────────────────────
def render_list_page():
    st.title("Flagship Registry")

    fc = st.session_state.filter_countries
    fs = st.session_state.filter_statuses
    fd = st.session_state.filter_divisions
    filters_active = bool(fc or fs or fd)

    search = st.text_input("Search", placeholder="Search flagships...")

    if _is_admin():
        with st.expander("🔍 Advanced filters", expanded=filters_active):
            cf1, cf2, cf3 = st.columns(3)
            with cf1:
                all_countries = [r["country"] for r in q("SELECT DISTINCT country FROM countries ORDER BY country")]
                st.session_state.filter_countries = st.multiselect("Country", all_countries, default=fc)
            with cf2:
                all_div = [r["division"] for r in q("SELECT DISTINCT division FROM flagships WHERE division IS NOT NULL AND division!='' ORDER BY division")]
                st.session_state.filter_divisions = st.multiselect("Division", all_div, default=fd)
            with cf3:
                all_st = [r["status"] for r in q("SELECT DISTINCT status FROM countries WHERE status IS NOT NULL AND status!='' ORDER BY status")]
                st.session_state.filter_statuses = st.multiselect("Status", all_st, default=fs)
            if st.button("♻️ Reset filters"):
                st.session_state.filter_countries = []
                st.session_state.filter_divisions = []
                st.session_state.filter_statuses = []
                st.rerun()
        fc = st.session_state.filter_countries
        fs = st.session_state.filter_statuses
        fd = st.session_state.filter_divisions
        filters_active = bool(fc or fs or fd)

    # Load flagships by permission
    if _is_admin():
        flagships = q("SELECT * FROM flagships ORDER BY rank")
    else:
        allowed = authorized_flagship_ids(_uname())
        if allowed:
            flagships = q("SELECT * FROM flagships WHERE id IN ({}) ORDER BY rank".format(
                ",".join(["?"] * len(allowed))), *allowed)
        else:
            flagships = []

    # Apply admin filters
    if _is_admin():
        if fc:
            flagships = [f for f in flagships if q1("""
                SELECT 1 FROM countries c JOIN use_cases uc ON c.use_case_id=uc.id
                WHERE uc.flagship_id=? AND c.country IN ({})""".format(",".join(["?"] * len(fc))), f["id"], *fc)]
        if fd:
            flagships = [f for f in flagships if f.get("division") in fd]
        if fs:
            flagships = [f for f in flagships if q1("""
                SELECT 1 FROM countries c JOIN use_cases uc ON c.use_case_id=uc.id
                WHERE uc.flagship_id=? AND c.status IN ({})""".format(",".join(["?"] * len(fs))), f["id"], *fs)]

    if search:
        s = search.lower()
        flagships = [f for f in flagships if any(s in str(v).lower() for v in f.values() if v is not None)]

    if _is_admin():
        if st.button("＋ New flagship", type="primary"):
            st.session_state.page = "flagship_form"
            st.session_state.edit_flagship_id = None
            st.rerun()

    st.divider()
    if not flagships:
        st.info("No accessible flagship.")
    else:
        # Inject CSS to make the card buttons look premium
        st.markdown("""
            <style>
            div[class*="st-key-card_"] {
                display: inline-block;
                width: 100%;
            }
            div[class*="st-key-card_"] button {
                display: flex;
                flex-direction: column;
                align-items: flex-start;
                justify-content: flex-start;
                text-align: left;
                width: 100% !important;
                height: 190px !important;
                padding: 16px !important;
                background-color: #ffffff !important;
                border: 1px solid #e5e7eb !important;
                border-left: 5px solid #2563eb !important;
                border-radius: 12px !important;
                box-shadow: 0 4px 6px -1px rgba(0, 0, 0, 0.05), 0 2px 4px -1px rgba(0, 0, 0, 0.03) !important;
                transition: all 0.2s cubic-bezier(0.4, 0, 0.2, 1) !important;
                white-space: normal !important;
                word-wrap: break-word !important;
                color: #1f2937 !important;
            }
            
            /* Hover effects */
            div[class*="st-key-card_"] button:hover {
                border-color: #2563eb !important;
                box-shadow: 0 10px 15px -3px rgba(0, 0, 0, 0.1), 0 4px 6px -2px rgba(0, 0, 0, 0.05) !important;
                transform: translateY(-4px) !important;
                background-color: #f9fafb !important;
            }
            
            /* Focus/Active state */
            div[class*="st-key-card_"] button:focus, div[class*="st-key-card_"] button:active {
                border-color: #2563eb !important;
                box-shadow: 0 0 0 2px rgba(37, 99, 235, 0.2) !important;
            }
            </style>
        """, unsafe_allow_html=True)

        cols_per_row = 3
        for i in range(0, len(flagships), cols_per_row):
            row_flagships = flagships[i:i+cols_per_row]
            cols = st.columns(cols_per_row)
            for idx, f in enumerate(row_flagships):
                with cols[idx]:
                    uc_n = q1("SELECT COUNT(*) n FROM use_cases WHERE flagship_id=?", f["id"])["n"]
                    val = q1("SELECT COALESCE(SUM(amount),0) n FROM value_entries ve JOIN use_cases uc ON ve.use_case_id=uc.id WHERE uc.flagship_id=?", f["id"])["n"]
                    
                    subtitle = f"📂 {uc_n} Use case{'s' if uc_n > 1 else ''}  ·  💰 {fmt_money(val)}"
                    if f.get("division"):
                        subtitle += f"  ·  🏢 {f['division']}"
                    
                    card_label = f"**#{f['rank']} — {f['name']}**\n\n{subtitle}"
                    if f.get("description"):
                        desc = f["description"].strip()
                        if len(desc) > 75:
                            desc = desc[:72] + "..."
                        card_label += f"\n\n*{desc}*"
                    
                    if st.button(card_label, key=f"card_{f['id']}", use_container_width=True):
                        st.session_state.page = "flagship_detail"
                        st.session_state.current_flagship_id = f["id"]
                        st.rerun()


# ── FLAGSHIP DETAIL ───────────────────────────────────────────────────────────
def render_flagship_detail_page():
    fid = st.session_state.get("current_flagship_id")
    f = q1("SELECT * FROM flagships WHERE id=?", fid)
    if not f:
        st.error("Flagship not found.")
        st.stop()
    if not is_flagship_authorized(_uname(), fid):
        st.error("Access denied.")
        st.stop()

    st.title(f["name"])
    st.caption(f"Flagship #{f['rank']}")
    if f.get("description"):
        st.write(f["description"])

    render_detail_grid(f, exclude_cols=["id", "name", "rank", "description"])
    st.write("")

    if _is_admin():
        ce, cd = st.columns(2)
        with ce:
            if st.button("✎ Edit flagship"):
                st.session_state.page = "flagship_form"
                st.session_state.edit_flagship_id = fid
                st.rerun()
        with cd:
            if st.button("🗑 Delete flagship"):
                delete_flagship_dialog(fid, f["name"])

    st.divider()
    st.subheader("Use cases")
    if _is_admin():
        if st.button("＋ Add use case", type="primary"):
            st.session_state.page = "usecase_form"
            st.session_state.edit_usecase_id = None
            st.rerun()

    use_cases = q("SELECT * FROM use_cases WHERE flagship_id=? ORDER BY id", fid)
    use_cases = [uc for uc in use_cases if is_usecase_authorized(_uname(), uc["id"])]
    if not use_cases:
        st.info("No accessible use case.")

    for uc in use_cases:
        n_ctry = q1("SELECT COUNT(*) n FROM countries WHERE use_case_id=?", uc["id"])["n"]
        uc_val = usecase_total_value(uc["id"])
        sub = f"{n_ctry} country(ies) · {fmt_money(uc_val)}"
        if uc.get("ai_act"):
            sub += f" · {uc['ai_act']}"
        card_open(uc["name"], sub, accent="#7c3aed")
        if uc.get("description"):
            st.caption(uc["description"])
        ca, cb = st.columns(2)
        with ca:
            if st.button("Open →", key=f"o_{uc['id']}"):
                st.session_state.page = "usecase_detail"
                st.session_state.current_usecase_id = uc["id"]
                st.rerun()
        with cb:
            if st.button("✎ Edit", key=f"e_{uc['id']}"):
                st.session_state.page = "usecase_form"
                st.session_state.edit_usecase_id = uc["id"]
                st.rerun()

    # Fetch use case IDs and country keys for history
    uc_rows = q("SELECT id FROM use_cases WHERE flagship_id=?", fid)
    child_records = []
    for uc in uc_rows:
        ucid = uc["id"]
        child_records.append(("use_cases", ucid))
        ctry_rows = q("SELECT country FROM countries WHERE use_case_id=?", ucid)
        for ctry in ctry_rows:
            child_records.append(("countries", f"{ucid}/{ctry['country']}"))

    render_local_history("flagships", fid, child_records)


# ── Typed column configuration (admin) ────────────────────────────────────────
def _render_column_config(table_name, cols):
    label = {"flagships": "Flagships", "use_cases": "Use Cases", "countries": "Countries"}[table_name]
    st.write("---")
    st.subheader(f"⚙️ Configure columns — {label}")
    st.warning("⚠️ Deleting a column permanently removes its data for all records.")

    with st.container(border=True):
        st.write("**Add a new field**")
        c1, c2 = st.columns(2)
        with c1:
            new_field = st.text_input("Field name", placeholder="e.g. budget, local_contact", key=f"new_field_name_{table_name}")
        with c2:
            ftype = st.selectbox("Field type", FIELD_TYPES, key=f"new_field_type_{table_name}",
                                 help="text = free text · select = dropdown with options · date = YYYY-MM-DD · number")
        options = ""
        if ftype == "select":
            options = st.text_input("Dropdown options (comma-separated)", placeholder="Option A, Option B, Option C", key=f"new_field_opts_{table_name}")
        if st.button("➕ Add field", type="primary", key=f"add_field_btn_{table_name}"):
            if not new_field:
                st.error("Please enter a name.")
            else:
                sanitized = sanitize_col_name(new_field)
                existing = get_table_columns(table_name)
                if not sanitized:
                    st.error("Invalid field name.")
                elif sanitized in existing:
                    st.error("This field already exists.")
                else:
                    try:
                        run(f"ALTER TABLE {table_name} ADD COLUMN {sanitized} TEXT")
                        add_custom_field(table_name, sanitized, ftype, options)
                        log_change(_uname(), "SCHEMA_ADD", table_name, sanitized, f"Added column {sanitized} ({ftype})")
                        st.success(f"Field '{sanitized}' ({ftype}) added!")
                        st.rerun()
                    except Exception as e:
                        st.error(f"Error adding field: {e}")

    st.write("**Custom fields**")
    deletable = [c for c in cols if c not in PROTECTED_COLS[table_name]]
    cfs = {cf["col_name"]: cf for cf in get_custom_fields(table_name)}
    if deletable:
        for c in deletable:
            l1, l2, l3 = st.columns([2, 2, 1])
            with l1:
                st.code(c)
            with l2:
                cf = cfs.get(c)
                st.caption(f"{cf['field_type']}" + (f" · {cf['options']}" if cf and cf['options'] else "") if cf else "text (native)")
            with l3:
                if st.button("🗑️", key=f"del_{table_name}_{c}"):
                    delete_custom_field_dialog(table_name, c)
    else:
        st.info("No custom field.")


# ── FLAGSHIP FORM ─────────────────────────────────────────────────────────────
def render_flagship_form_page():
    fid = st.session_state.get("edit_flagship_id")
    f = q1("SELECT * FROM flagships WHERE id=?", fid) if fid else {}
    is_new = not bool(f)
    if fid and not is_flagship_authorized(_uname(), fid):
        st.error("Access denied.")
        st.stop()

    st.title("New flagship" if is_new else "Edit flagship")
    cols = get_table_columns("flagships")
    extra = [c for c in cols if c not in ("id", "name", "rank")]

    fv = {}
    with st.form("f_form"):
        name = st.text_input("Name *", value=f.get("name", ""))
        rank = st.number_input("Rank", min_value=1, max_value=99, value=int(f.get("rank", 1)))
        if extra:
            c1, c2 = st.columns(2)
            for i, cn in enumerate(extra):
                with c1 if i % 2 == 0 else c2:
                    fv[cn] = render_field_input("flagships", cn, f.get(cn, ""), "fs")
        if st.form_submit_button("💾 Save", type="primary"):
            if not name:
                st.error("Name is required.")
            else:
                new_id = fid or next_f_id()
                data = {"id": new_id, "name": name, "rank": rank}
                data.update({cn: fv[cn] for cn in extra})
                if is_new:
                    ks = list(data.keys())
                    run(f"INSERT INTO flagships ({', '.join(ks)}) VALUES ({', '.join(['?']*len(ks))})", *data.values())
                    log_change(_uname(), "INSERT", "flagships", new_id, f"Created flagship: {name}")
                else:
                    log_record_update("flagships", new_id, data, "SELECT * FROM flagships WHERE id=?", new_id)
                    ks = [k for k in data if k != "id"]
                    run(f"UPDATE flagships SET {', '.join(k+'=?' for k in ks)} WHERE id=?", *[data[k] for k in ks], new_id)
                st.session_state.current_flagship_id = new_id
                st.session_state.page = "flagship_detail"
                st.rerun()

    if _is_admin():
        _render_column_config("flagships", cols)


# ── USE CASE FORM ─────────────────────────────────────────────────────────────
def render_usecase_form_page():
    ucid = st.session_state.get("edit_usecase_id")
    uc = q1("SELECT * FROM use_cases WHERE id=?", ucid) if ucid else {}
    fid = uc.get("flagship_id") if uc else st.session_state.get("current_flagship_id")
    is_new = not bool(uc)
    if ucid and not is_usecase_authorized(_uname(), ucid):
        st.error("Access denied.")
        st.stop()

    st.title("New use case" if is_new else "Edit use case")
    cols = get_table_columns("use_cases")
    extra = [c for c in cols if c not in ("id", "flagship_id", "name")]

    fv = {}
    with st.form("uc_form"):
        name = st.text_input("Name *", value=uc.get("name", ""))
        if extra:
            c1, c2 = st.columns(2)
            for i, cn in enumerate(extra):
                with c1 if i % 2 == 0 else c2:
                    fv[cn] = render_field_input("use_cases", cn, uc.get(cn, ""), "uc")
        if st.form_submit_button("💾 Save", type="primary"):
            if not name:
                st.error("Name is required.")
            else:
                new_id = ucid or next_uc_id(fid)
                data = {"id": new_id, "flagship_id": fid, "name": name}
                data.update({cn: fv[cn] for cn in extra})
                if is_new:
                    ks = list(data.keys())
                    run(f"INSERT INTO use_cases ({', '.join(ks)}) VALUES ({', '.join(['?']*len(ks))})", *data.values())
                    log_change(_uname(), "INSERT", "use_cases", new_id, f"Created use case: {name}")
                else:
                    log_record_update("use_cases", new_id, data, "SELECT * FROM use_cases WHERE id=?", new_id)
                    ks = [k for k in data if k != "id"]
                    run(f"UPDATE use_cases SET {', '.join(k+'=?' for k in ks)} WHERE id=?", *[data[k] for k in ks], new_id)
                st.session_state.current_usecase_id = new_id
                st.session_state.page = "usecase_detail"
                st.rerun()

    if _is_admin():
        _render_column_config("use_cases", cols)


# ── USE CASE DETAIL ───────────────────────────────────────────────────────────
def render_usecase_detail_page():
    ucid = st.session_state.get("current_usecase_id")
    uc = q1("SELECT * FROM use_cases WHERE id=?", ucid)
    if not uc:
        st.error("Use case not found.")
        st.stop()
    if not is_usecase_authorized(_uname(), ucid):
        st.error("Access denied.")
        st.stop()

    st.title(uc["name"])
    st.caption(uc["id"])
    if uc.get("description"):
        st.write(uc["description"])

    render_detail_grid(uc, exclude_cols=["id", "flagship_id", "name", "description"])
    st.write("")

    # IA value consolidated + breakdown by nature
    st.subheader("💰 IA Value (consolidated)")
    by_nature = usecase_value_by_nature(ucid)
    total = sum(by_nature.values())
    if total == 0:
        st.info("No IA value recorded for this use case.")
    else:
        st.metric("Total IA Value", fmt_money(total))
        cols_n = st.columns(min(len(by_nature), 5) or 1)
        for i, (nat, amt) in enumerate(by_nature.items()):
            cols_n[i % len(cols_n)].metric(nat, fmt_money(amt))

    st.write("")
    if _is_admin():
        ce, cd = st.columns(2)
        with ce:
            if st.button("✎ Edit use case"):
                st.session_state.page = "usecase_form"
                st.session_state.edit_usecase_id = ucid
                st.rerun()
        with cd:
            if st.button("🗑 Delete use case"):
                delete_usecase_dialog(ucid, uc["name"])

    st.divider()
    st.subheader("Country deployments")

    if _is_admin():
        if st.button("＋ Add country", type="primary"):
            st.session_state.page = "country_form"
            st.session_state.edit_country = None
            st.rerun()

    # Year filter for deployments
    years = [r["year"] for r in q("SELECT DISTINCT year FROM value_entries WHERE use_case_id=? ORDER BY year", ucid)]
    sel_year = None
    if years:
        opt = ["All years"] + [str(y) for y in years]
        choice = st.selectbox("Filter by year", opt, key="uc_year_filter")
        sel_year = None if choice == "All years" else int(choice)

    countries = q("SELECT * FROM countries WHERE use_case_id=? ORDER BY country", ucid)
    countries = [c for c in countries if is_country_authorized(_uname(), c["id"])]
    if not countries:
        st.info("No accessible country deployment.")

    for c in countries:
        cval = country_total_value(ucid, c["country"])
        card_open(c["country"],
                  f"{c.get('status') or '—'} · {fmt_money(cval)} · prod {c.get('prod_date') or '—'}",
                  accent="#0891b2")
        # Value entries (optionally year-filtered)
        entries = get_value_entries(use_case_id=ucid, country=c["country"], year=sel_year)
        if entries:
            if _is_admin():
                # admin: show quarterly detail
                try:
                    import pandas as pd
                    rows_df = []
                    for e in entries:
                        qs = split_annual_to_quarters(e["amount"], c["prod_date"], e["year"])
                        rows_df.append({"Year": e["year"], "Nature": e["nature"], "Annual": e["amount"],
                                        "Q1": qs[0], "Q2": qs[1], "Q3": qs[2], "Q4": qs[3]})
                    df = pd.DataFrame(rows_df)
                    st.dataframe(df.style.format({c2: "{:,.0f}".format for c2 in ["Annual", "Q1", "Q2", "Q3", "Q4"]}),
                                 use_container_width=True, hide_index=True)
                except Exception:
                    for e in entries:
                        st.caption(f"{e['year']} · {e['nature']}: {fmt_money(e['amount'])}")
            else:
                # normal user: annual values only
                for yr in sorted({e["year"] for e in entries}):
                    parts = [f"{e['nature']}: {fmt_money(e['amount'])}" for e in entries if e["year"] == yr]
                    st.caption(f"**{yr}** — " + "  |  ".join(parts))

        cm, cd2 = st.columns(2)
        with cm:
            if st.button("✎ Edit / values", key=f"mc_{c['id']}"):
                st.session_state.page = "country_form"
                st.session_state.edit_country = c["country"]
                st.rerun()
        if _is_admin():
            with cd2:
                if st.button("🗑 Delete", key=f"dc_{c['id']}"):
                    delete_country_dialog(ucid, c["country"])

    # Fetch country keys for history
    ctry_rows = q("SELECT country FROM countries WHERE use_case_id=?", ucid)
    child_records = [("countries", f"{ucid}/{c['country']}") for c in ctry_rows]

    render_local_history("use_cases", ucid, child_records)


# ── COUNTRY FORM (+ IA value editor) ──────────────────────────────────────────
def render_country_form_page():
    ucid = st.session_state.get("current_usecase_id")
    edit_c = st.session_state.get("edit_country")
    c_data = q1("SELECT * FROM countries WHERE use_case_id=? AND country=?", ucid, edit_c) if edit_c else {}
    is_new = not bool(c_data)
    used = [r["country"] for r in q("SELECT country FROM countries WHERE use_case_id=?", ucid)]
    available = [c for c in COUNTRIES if c not in used or c == edit_c]

    if edit_c and not _is_admin() and c_data and not is_country_authorized(_uname(), c_data["id"]):
        st.error("Access denied.")
        st.stop()

    st.title("Add country" if is_new else f"Edit: {edit_c}")
    cols = get_table_columns("countries")
    # normal users can only see/set a limited set; admins see all
    if _is_admin():
        extra = [c for c in cols if c not in ("id", "use_case_id", "country")]
    else:
        extra = [c for c in cols if c in ("status", "prod_date", "owner", "mrr")]

    fv = {}
    with st.form("c_form"):
        st.write("**Deployment**")
        country = st.selectbox("Country *", available,
                               index=available.index(edit_c) if edit_c in available else 0, disabled=not is_new)
        if extra:
            c1, c2 = st.columns(2)
            for i, cn in enumerate(extra):
                with c1 if i % 2 == 0 else c2:
                    fv[cn] = render_field_input("countries", cn, c_data.get(cn, ""), "co")
        if st.form_submit_button("💾 Save", type="primary"):
            data = {"use_case_id": ucid, "country": country}
            data.update({cn: fv[cn] for cn in extra})
            if is_new:
                ks = list(data.keys())
                run(f"INSERT INTO countries ({', '.join(ks)}) VALUES ({', '.join(['?']*len(ks))})", *data.values())
                log_change(_uname(), "INSERT", "countries", f"{ucid}/{country}", f"Created deployment {country}")
            else:
                log_record_update("countries", f"{ucid}/{edit_c}", data, "SELECT * FROM countries WHERE use_case_id=? AND country=?", ucid, edit_c)
                ks = [k for k in data if k not in ("use_case_id", "country")]
                run(f"UPDATE countries SET {', '.join(k+'=?' for k in ks)} WHERE use_case_id=? AND country=?",
                    *[data[k] for k in ks], ucid, edit_c)
            st.session_state.edit_country = country
            st.rerun()

    # ── IA value editor ───────────────────────────────────────────────────────
    target = edit_c or st.session_state.get("edit_country")
    if target and q1("SELECT 1 FROM countries WHERE use_case_id=? AND country=?", ucid, target):
        st.divider()
        st.subheader("💰 IA Value")
        prod_date = q1("SELECT prod_date FROM countries WHERE use_case_id=? AND country=?", ucid, target)["prod_date"]

        existing = get_value_entries(use_case_id=ucid, country=target)
        if existing:
            st.write("**Recorded values**")
            for e in existing:
                if _is_admin():
                    qs = split_annual_to_quarters(e["amount"], prod_date, e["year"])
                    line = (f"**{e['year']}** · {e['nature']} : {fmt_money(e['amount'])} → "
                            f"Q1 {fmt_money(qs[0])} · Q2 {fmt_money(qs[1])} · Q3 {fmt_money(qs[2])} · Q4 {fmt_money(qs[3])}")
                else:
                    line = f"**{e['year']}** · {e['nature']} : {fmt_money(e['amount'])} (annual)"
                ca, cb = st.columns([5, 1])
                with ca:
                    st.write(line)
                with cb:
                    if st.button("🗑", key=f"dv_{e['id']}"):
                        delete_value_entry_dialog(e["id"], e["year"], e["nature"], e["amount"], target)

        st.write("**Add / update an annual value**")
        if not _is_admin():
            st.caption("Enter the annual value; quarterly split is computed automatically from the production date.")
        with st.form("val_form"):
            v1, v2, v3 = st.columns(3)
            with v1:
                yr = st.number_input("Year", min_value=2020, max_value=2035, value=2026)
            with v2:
                nat = st.selectbox("Nature", VALUE_NATURES)
            with v3:
                amt = st.number_input("Annual value (k€)", min_value=0, value=0)
            if st.form_submit_button("💾 Save value", type="primary"):
                run("""INSERT INTO value_entries (use_case_id, country, year, nature, amount)
                       VALUES (?,?,?,?,?)
                       ON CONFLICT(use_case_id, country, year, nature)
                       DO UPDATE SET amount=excluded.amount""",
                    ucid, target, int(yr), nat, float(amt))
                log_change(_uname(), "UPSERT", "value_entries", f"{ucid}/{target}/{yr}/{nat}", f"{nat} {yr} = {amt} k€ ({target})")
                qs = split_annual_to_quarters(amt, prod_date, int(yr))
                st.success(f"Saved. {yr} split → Q1 {fmt_money(qs[0])} / Q2 {fmt_money(qs[1])} / Q3 {fmt_money(qs[2])} / Q4 {fmt_money(qs[3])}")
                st.rerun()

    if _is_admin():
        _render_column_config("countries", cols)


# ── USERS ADMIN (3-level permissions) ─────────────────────────────────────────
def _perm_editor(username, key_prefix):
    """Render the 3-level permission editor and return the chosen perms list."""
    current = get_user_perms(username)
    cur_flag = {p["ref_id"] for p in current if p["scope"] == "flagship"}
    cur_uc = {p["ref_id"] for p in current if p["scope"] == "usecase"}
    cur_ctry = {p["ref_id"] for p in current if p["scope"] == "country"}

    flagships = q("SELECT id, rank, name FROM flagships ORDER BY rank")
    fl_opts = {f["id"]: f"#{f['rank']} {f['name']}" for f in flagships}

    st.caption("Grant access at any level. A flagship grant covers all its use cases and countries; "
               "a use case grant covers all its countries. Use the levels below to mix broad and precise access.")

    # Level 1: flagships (all or specific)
    sel_flag = st.multiselect("① Flagships (full access incl. all use cases & countries)",
                              list(fl_opts.keys()), default=[f for f in cur_flag if f in fl_opts],
                              format_func=lambda x: fl_opts[x], key=f"{key_prefix}_flag")

    # Level 2: use cases (only those NOT already covered by a selected flagship)
    uc_rows = q("SELECT id, name, flagship_id FROM use_cases ORDER BY id")
    uc_opts = {u["id"]: f"{fl_opts.get(u['flagship_id'],'')} ▸ {u['name']}" for u in uc_rows
               if u["flagship_id"] not in sel_flag}
    sel_uc = st.multiselect("② Use cases (full access incl. all its countries)",
                            list(uc_opts.keys()),
                            default=[u for u in cur_uc if u in uc_opts],
                            format_func=lambda x: uc_opts[x], key=f"{key_prefix}_uc")

    # Level 3: countries (only those not covered above)
    covered_uc = set(sel_uc)
    ctry_rows = q("""SELECT c.id, c.country, uc.name uc_name, uc.id uc_id, uc.flagship_id
                     FROM countries c JOIN use_cases uc ON c.use_case_id = uc.id ORDER BY c.id""")
    ctry_opts = {str(c["id"]): f"{fl_opts.get(c['flagship_id'],'')} ▸ {c['uc_name']} ▸ {c['country']}"
                 for c in ctry_rows if c["flagship_id"] not in sel_flag and c["uc_id"] not in covered_uc}
    sel_ctry = st.multiselect("③ Specific countries", list(ctry_opts.keys()),
                              default=[c for c in cur_ctry if c in ctry_opts],
                              format_func=lambda x: ctry_opts[x], key=f"{key_prefix}_ctry")

    perms = ([("flagship", f) for f in sel_flag] +
             [("usecase", u) for u in sel_uc] +
             [("country", c) for c in sel_ctry])
    return perms


def render_users_admin_page():
    if not _is_admin():
        st.error("Admin only.")
        st.stop()

    st.title("⚙️ Access Management")
    st.subheader("👥 Users & permissions")

    st.write("### ➕ Create a user")
    new_u = st.text_input("Username", key="adm_new_u")
    new_p = st.text_input("Password", type="password", key="adm_new_p")
    make_admin = st.checkbox("👑 This user is an administrator", key="adm_new_admin")
    st.caption("An administrator has full access (dashboard, configuration) without per-scope grants.")

    perms = []
    if not make_admin:
        with st.container(border=True):
            perms = _perm_editor("__new__", "new")

    if st.button("Create user", type="primary"):
        if not new_u or not new_p:
            st.error("Please fill username and password.")
        elif new_u == "admin":
            st.error("'admin' is reserved.")
        elif create_user(new_u, new_p, is_admin=make_admin):
            if not make_admin:
                set_user_perms(new_u, perms)
            log_change(_uname(), "INSERT", "users", new_u,
                       f"Created {'admin' if make_admin else 'standard'} user: {new_u}")
            st.success(f"User '{new_u}' created!")
            st.rerun()
        else:
            st.error("This username already exists.")

    st.divider()
    st.write("### 👥 Existing users")
    users = q("SELECT username, is_admin FROM users WHERE username != 'admin' ORDER BY username")
    if not users:
        st.info("No other user.")
    for u in users:
        uname = u["username"]
        badge = "👑 Admin" if u["is_admin"] else "👤 Standard"
        with st.expander(f"{badge} — {uname}"):
            is_adm = st.checkbox("👑 Administrator", value=bool(u["is_admin"]), key=f"adm_{uname}")
            new_perms = []
            if not is_adm:
                new_perms = _perm_editor(uname, f"edit_{uname}")
            cs, cd = st.columns(2)
            with cs:
                if st.button("💾 Save", key=f"save_{uname}"):
                    run("UPDATE users SET is_admin=? WHERE username=?", 1 if is_adm else 0, uname)
                    if is_adm:
                        set_user_perms(uname, [])
                    else:
                        set_user_perms(uname, new_perms)
                    log_change(_uname(), "UPDATE", "users", uname, f"Updated permissions (admin={is_adm})")
                    st.success("Saved!")
                    st.rerun()
            with cd:
                if st.button("🗑️ Delete", key=f"del_{uname}", type="primary"):
                    delete_user_dialog(uname)


# ── AUDIT LOG ─────────────────────────────────────────────────────────────────
def render_audit_history_page():
    if not _is_admin():
        st.error("Admin only.")
        st.stop()

    st.title("📜 Activity Log")
    st.subheader("Audit trail of platform changes")

    col_clear, _ = st.columns([1, 3])
    with col_clear:
        if st.button("🗑️ Clear log", type="primary"):
            clear_audit_logs_dialog()

    st.write("### 🔍 Filter")
    c1, c2, c3 = st.columns(3)
    with c1:
        authors = [r["username"] for r in q("SELECT DISTINCT username FROM audit_logs ORDER BY username")]
        f_auth = st.multiselect("Author", authors)
    with c2:
        actions = [r["action"] for r in q("SELECT DISTINCT action FROM audit_logs ORDER BY action")]
        f_act = st.multiselect("Action", actions)
    with c3:
        tables = [r["table_name"] for r in q("SELECT DISTINCT table_name FROM audit_logs ORDER BY table_name")]
        f_tab = st.multiselect("Table", tables)

    sql, params, where = "SELECT * FROM audit_logs", [], []
    if f_auth:
        where.append("username IN ({})".format(",".join(["?"] * len(f_auth)))); params += f_auth
    if f_act:
        where.append("action IN ({})".format(",".join(["?"] * len(f_act)))); params += f_act
    if f_tab:
        where.append("table_name IN ({})".format(",".join(["?"] * len(f_tab)))); params += f_tab
    if where:
        sql += " WHERE " + " AND ".join(where)
    sql += " ORDER BY timestamp DESC"
    logs = q(sql, *params)

    st.write("---")
    if not logs:
        st.info("No matching entry.")
    else:
        emojis = {"INSERT": "➕", "UPDATE": "📝", "UPSERT": "💾", "DELETE": "🗑️", "SCHEMA_ADD": "⚙️➕", "SCHEMA_DROP": "⚙️🗑️"}
        for log in logs:
            e = emojis.get(log["action"], "🔔")
            st.markdown(
                f"**{e} {log['action']}** — `{log['table_name']}`  ·  👤 {log['username']}  ·  📅 {log['timestamp']}  \n"
                f"Ref `{log['record_id']}` — {log['details']}")
            st.write("---")
